# ============================================================================
# MEXICAN HAT FILTERED VARIANCE ANALYSIS FOR AGN LIGHT CURVES
# Based on Arévalo et al. (2023, MNRAS) and Arévalo et al. (2012, MNRAS)
# ============================================================================

The **Mexican Hat (MH) filter** is a zero-mean wavelet that acts as a bandpass
filter in the time domain. For each timescale $\sigma$ (in rest-frame days),
the method estimates the *filtered variance* of the light curve:

1. Convolve the light curve with the MH kernel of width $\sigma$.
2. Only evaluate the convolution at times where the kernel is
   sufficiently well covered by data (robust against gaps).
3. Compute the variance of the filtered signal.
4. Subtract the contribution from observational noise (if `flux_err` exists).

The result, $\mathrm{Var}(\sigma)$, is a discrete proxy for the power spectral density (PSD):
it measures the power in a frequency band around

$$
f \sim \frac{1}{2\pi\sigma}.
$$

If the PSD has a break at some timescale, the slope of
$\log(\mathrm{Var})$ vs. $\log(\sigma)$ changes around that break.

## Key references

- Arévalo et al. (2012), *MNRAS*, **426**, 1793 — the original MH method.
- Arévalo et al. (2023), *MNRAS*, **526**, 1250 — application to quasars and discovery of the "universal PSD" dependent on $M_{\mathrm{BH}}$ and Eddington ratio.

# ============================================================================

In [11]:
# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------

base_dir      = r"ztffp_final_data_2025"
csv_meta_path = r"C:\Users\vitma\Documents\Estudos\UFRGS! DO IT!\Projeto Thaisa Rogério\wu_shen_selected_columns.csv"
BANDS_TO_PROCESS = ['g', 'r']

# Scales in REST-FRAME days. These are the "sigma" of the MH kernel.
# Arévalo 2023 used [30, 75, 150, 300]. We add extra scales for finer sampling.
SCALES_REST_DAYS = [20, 30, 50, 75, 100, 150, 200, 300, 450, 600]

# Validity criteria (see mh_variance_single for details)
MIN_POINTS_IN_KERNEL = 5     # minimum points within 3*sigma of t_i
MIN_POINTS_PER_SIDE  = 2     # minimum points on each side within 2*sigma
MIN_SPAN_FRACTION    = 0.4   # minimum span of points within kernel, in units of 6*sigma

# Physical filters (identical to the existing Petrecca pipeline)
Z_MIN, Z_MAX = 0.4, 0.5
mass_edges = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
edd_edges  = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5]

# Noise subtraction: number of Gaussian noise realizations per object/scale
N_NOISE_SIMULATIONS = 20

OUT_DIR = "mexican_hat_results"
os.makedirs(OUT_DIR, exist_ok=True)


In [12]:
# ----------------------------------------------------------------------------
# UTILITY: Mexican Hat kernel and variance at a single scale
# ----------------------------------------------------------------------------

def mexican_hat_kernel(u, sigma):
    """
    Mexican Hat (Ricker) wavelet, unnormalized.

    K(u) = (1 - u^2/sigma^2) * exp(-u^2/(2*sigma^2))

    Properties:
      - Zero mean over (-inf, +inf)
      - Positive central lobe for |u| < sigma
      - Negative side lobes for sigma < |u| < ~3*sigma
      - Zero crossing at |u| = sigma
    """
    x = u / sigma
    return (1.0 - x**2) * np.exp(-0.5 * x**2)


def mh_variance_single(t, f, sigma,
                       min_points=MIN_POINTS_IN_KERNEL,
                       min_per_side=MIN_POINTS_PER_SIDE,
                       min_span_frac=MIN_SPAN_FRACTION):
    """
    Compute the Mexican Hat filtered variance at a single scale.

    Parameters
    ----------
    t : array, shape (n,)
        Observation times, sorted ascending (rest-frame days).
    f : array, shape (n,)
        Normalized flux values (dimensionless).
    sigma : float
        Filter width in days (same units as t).
    min_points, min_per_side, min_span_frac : float
        Validity thresholds (see module docstring).

    Returns
    -------
    var : float
        Variance of the filtered signal (NaN if insufficient data).
    n_valid : int
        Number of valid points used.
    """
    n = len(t)
    if n < min_points:
        return np.nan, 0

    # Pairwise time differences: dt[i, j] = t_i - t_j
    dt = t[:, None] - t[None, :]

    # Mexican Hat kernel at each pair (unnormalized)
    K = mexican_hat_kernel(dt, sigma)

    # Kernel support mask: only |dt| < 3*sigma contributes (beyond that, K ~ 0)
    support = np.abs(dt) < 3.0 * sigma

    # For each test point t_i, the normalization is sqrt(sum_j K_ij^2 M_j).
    # This ensures that for white noise with per-point variance sigma_n^2,
    # the filtered value F(t_i) has variance ~ sigma_n^2 (scale-independent).
    K2 = (K ** 2) * support
    norm_sq = np.sum(K2, axis=1)

    # ---- Validity mask ----
    # A point t_i is "valid" if the kernel is well covered by data:
    #   (a) at least `min_points` data points within 3*sigma
    #   (b) at least `min_per_side` points on each side within 2*sigma
    #   (c) the data span within the kernel covers at least 4*sigma*min_span_frac
    #       (i.e., the kernel is not truncated by gaps or by the light-curve edges)
    valid = np.zeros(n, dtype=bool)

    for i in range(n):
        if norm_sq[i] <= 0:
            continue
        sup_i = support[i]
        n_within = np.sum(sup_i)
        if n_within < min_points:
            continue

        dt_row = dt[i]
        n_left  = np.sum((dt_row > 0) & (dt_row < 2.0 * sigma) & sup_i)
        n_right = np.sum((dt_row < 0) & (dt_row > -2.0 * sigma) & sup_i)
        if n_left < min_per_side or n_right < min_per_side:
            continue

        t_in = t[sup_i]
        span = t_in.max() - t_in.min() if len(t_in) > 0 else 0.0
        if span < 4.0 * sigma * min_span_frac:
            continue

        valid[i] = True

    n_valid = int(np.sum(valid))
    if n_valid < max(5, min_points):
        return np.nan, n_valid

    # ---- Filtered values ----
    # F(t_i) = sum_j f_j K_ij M_j / sqrt(sum_j K_ij^2 M_j)
    idx_valid = np.where(valid)[0]
    F = np.empty(n_valid, dtype=float)
    for k, i in enumerate(idx_valid):
        numer = np.sum(f * K[i] * support[i])
        F[k] = numer / np.sqrt(norm_sq[i])

    # Variance (mean of squares; F has zero mean by construction)
    var = np.mean(F ** 2) - np.mean(F) ** 2
    return var, n_valid


def mh_variance_with_noise(t, f, f_err, sigma, n_sim=N_NOISE_SIMULATIONS):
    """
    Compute MH variance at a single scale, subtracting the noise contribution.

    Noise variance is estimated by generating n_sim Gaussian noise realizations
    with per-point standard deviation f_err, applying the MH filter to each,
    and taking the median variance. This is the procedure of Arévalo 2023
    (Appendix B).

    If f_err is None (no flux errors available), returns the raw variance
    without subtraction and flags it with var_noise = 0.

    Returns
    -------
    var_corrected : float
        Noise-subtracted variance (NaN if negative or unreliable).
    var_observed : float
        Raw variance (signal + noise).
    var_noise : float
        Estimated noise contribution.
    n_valid : int
    """
    var_obs, n_valid = mh_variance_single(t, f, sigma)

    if np.isnan(var_obs) or f_err is None:
        return var_obs, var_obs, 0.0, n_valid
    if np.all(f_err == 0) or np.all(np.isnan(f_err)):
        return var_obs, var_obs, 0.0, n_valid

    rng = np.random.default_rng(42)
    noise_vars = []
    for _ in range(n_sim):
        noise = rng.normal(0.0, f_err)
        v_n, _ = mh_variance_single(t, noise, sigma)
        if not np.isnan(v_n):
            noise_vars.append(v_n)

    if len(noise_vars) == 0:
        return var_obs, var_obs, 0.0, n_valid

    var_noise = float(np.median(noise_vars))
    var_corr = var_obs - var_noise
    if var_corr < 0:
        var_corr = np.nan  # noise-dominated measurement; unreliable

    return var_corr, var_obs, var_noise, n_valid


def compute_mh_spectrum(t, f, f_err, scales):
    """
    Compute MH variance at all requested scales.

    Returns a dict with arrays over `scales`.
    """
    idx = np.argsort(t)
    t, f = t[idx], f[idx]
    if f_err is not None:
        f_err = f_err[idx]

    out = {
        'scales':        [],
        'variance':      [],
        'variance_err':  [],
        'variance_obs':  [],
        'variance_noise': [],
        'n_valid':       [],
    }

    for sigma in scales:
        v_corr, v_obs, v_noise, nv = mh_variance_with_noise(t, f, f_err, sigma)
        out['scales'].append(sigma)
        out['variance'].append(v_corr)
        out['variance_obs'].append(v_obs)
        out['variance_noise'].append(v_noise)
        out['n_valid'].append(nv)
        # Approximate 1-sigma uncertainty on the variance:
        # Var[sigma^2_hat] ~ 2*sigma^4/N for Gaussian fluctuations.
        out['variance_err'].append(
            np.nan if (np.isnan(v_corr) or nv <= 1) else v_corr * np.sqrt(2.0 / nv)
        )
    return out


# ----------------------------------------------------------------------------
# MAIN LOOP: process all objects for one band
# ----------------------------------------------------------------------------

def process_band(band, base_dir, csv_meta_path,
                 z_min=Z_MIN, z_max=Z_MAX,
                 mass_edges=mass_edges, edd_edges=edd_edges,
                 scales=SCALES_REST_DAYS):
    """
    Process all objects in `base_dir` for the given band, apply physical filters,
    compute the Mexican Hat variance spectrum, and return a DataFrame.
    """
    print(f"\n{'='*60}\n Processing band: {band.upper()}\n{'='*60}")

    # ---- Load metadata ----
    print("Loading metadata...")
    meta = pd.read_csv(csv_meta_path.replace("\\", "/"))
    ra_cat    = meta['RA'].values
    dec_cat   = meta['DEC'].values
    logmbh    = meta['LOGMBH'].values
    logedd    = meta['LOGLEDD_RATIO'].values
    z_col     = 'Z_DR16Q' if 'Z_DR16Q' in meta.columns else 'Z'
    z_cat     = meta[z_col].values

    folders = [f for f in os.listdir(base_dir)
               if os.path.isdir(os.path.join(base_dir, f))
               and "plots" not in f and "output" not in f]

    print(f"Found {len(folders)} folders to scan.")
    results = []

    for folder in tqdm(folders, desc=f"Band {band}"):
        # ---- Crossmatch by RA/DEC ----
        try:
            ra_f, dec_f = map(float, folder.split('_'))
        except ValueError:
            continue

        dist = np.sqrt((ra_cat - ra_f) ** 2 + (dec_cat - dec_f) ** 2)
        idx = int(np.argmin(dist))
        if dist[idx] > 1e-4:
            continue

        m, e, z = logmbh[idx], logedd[idx], z_cat[idx]
        if np.isnan(m) or np.isnan(e) or np.isnan(z):
            continue
        if not (z_min <= z <= z_max):
            continue
        if not (mass_edges[0] <= m < mass_edges[-1]):
            continue
        if not (edd_edges[0] <= e < edd_edges[-1]):
            continue

        # ---- Read light curve ----
        txt_files = glob.glob(os.path.join(base_dir, folder,
                                           f"*_ztf_{band}_filtered.txt"))
        if not txt_files:
            continue
        try:
            df = pd.read_csv(txt_files[0])
        except Exception:
            continue
        if len(df) < 10 or 'mjd' not in df.columns or 'flux' not in df.columns:
            continue

        df = df.sort_values('mjd')
        t_obs = df['mjd'].values
        f_raw = df['flux'].values
        f_err_raw = None
        for col in ('flux_err', 'flux_error', 'err'):
            if col in df.columns:
                f_err_raw = df[col].values
                break

        # ---- Normalize by median flux ----
        med = np.median(f_raw)
        if med <= 0 or np.isnan(med):
            continue
        f_norm = (f_raw - med) / med
        f_err_norm = (f_err_raw / med) if f_err_raw is not None else None

        # ---- Convert to rest frame ----
        t_rest = t_obs / (1.0 + z)
        t_rest = t_rest - t_rest.min()

        # ---- Mexican Hat variance spectrum ----
        spec = compute_mh_spectrum(t_rest, f_norm, f_err_norm, scales)

        results.append({
            'ra': ra_f, 'dec': dec_f,
            'logmbh': m, 'logedd': e, 'z': z,
            'n_epochs': len(df),
            't_baseline_rest': float(t_rest.max()),
            **spec,
        })

    print(f"Processed {len(results)} objects for band {band}.")
    return pd.DataFrame(results)


# ----------------------------------------------------------------------------
# STACKING: median variance in bins of mass (or Eddington)
# ----------------------------------------------------------------------------

def stack_variance(df, mass_edges, edd_edges, group_by='mass', min_objects=3):
    """
    For each bin in mass (or Eddington), compute the median variance at each scale.

    Returns a dict: {bin_label: {scales, median_variance, n_objects, bin_center}}.
    """
    out = {}
    if group_by == 'mass':
        edges, col = mass_edges, 'logmbh'
        fmt = "logM {:.1f}-{:.1f}"
    elif group_by == 'edd':
        edges, col = edd_edges, 'logedd'
        fmt = "logλ {:.1f}-{:.1f}"
    else:
        raise ValueError("group_by must be 'mass' or 'edd'")

    scales = list(df['scales'].iloc[0])
    for k in range(len(edges) - 1):
        lo, hi = edges[k], edges[k + 1]
        sub = df[(df[col] >= lo) & (df[col] < hi)]
        if len(sub) < min_objects:
            continue

        med_var = []
        for j, s in enumerate(scales):
            vals = [row['variance'][j] for _, row in sub.iterrows()
                    if not np.isnan(row['variance'][j]) and row['variance'][j] > 0]
            med_var.append(np.median(vals) if len(vals) >= min_objects else np.nan)

        label = fmt.format(lo, hi)
        out[label] = {
            'scales': scales,
            'median_variance': med_var,
            'n_objects': len(sub),
            'bin_center': 0.5 * (lo + hi),
        }
    return out


# ----------------------------------------------------------------------------
# PLOTTING
# ----------------------------------------------------------------------------

def plot_variance_spectra(stacked, title, output_path):
    """Plot log(Var) vs log(sigma) for each bin in the stacked dict."""
    fig, ax = plt.subplots(figsize=(9, 6))
    cmap = plt.get_cmap('viridis')
    n = len(stacked)

    for i, (label, d) in enumerate(stacked.items()):
        color = cmap(i / max(1, n - 1))
        s = np.array(d['scales'], dtype=float)
        v = np.array(d['median_variance'], dtype=float)
        ok = (~np.isnan(v)) & (v > 0)
        if np.sum(ok) < 2:
            continue
        ax.plot(s[ok], v[ok], marker='o', lw=2, ms=6, color=color,
                label=f"{label} (N={d['n_objects']})")

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Rest-frame timescale $\sigma$ [days]', fontsize=13, fontweight='bold')
    ax.set_ylabel(r'Median filtered variance', fontsize=13, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(ls=':', alpha=0.5)
    ax.legend(fontsize=9, loc='best', framealpha=0.9)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    print(f"  saved {output_path}")


def plot_individual_example(row, output_path):
    """Plot variance spectrum of a single object (signal, noise, corrected)."""
    fig, ax = plt.subplots(figsize=(8, 5))
    s   = np.array(row['scales'], dtype=float)
    v   = np.array(row['variance'], dtype=float)
    verr= np.array(row['variance_err'], dtype=float)
    vo  = np.array(row['variance_obs'], dtype=float)
    vn  = np.array(row['variance_noise'], dtype=float)

    ok = (~np.isnan(v)) & (v > 0)
    ax.errorbar(s[ok], v[ok], yerr=verr[ok], fmt='o-', lw=2, ms=7,
                capsize=3, color='navy', label='Corrected variance')
    ax.plot(s, vo, 's--', alpha=0.6, color='crimson', label='Raw variance')
    ax.plot(s, vn, '^:',  alpha=0.6, color='gray',    label='Noise contribution')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Rest-frame timescale [days]', fontsize=12, fontweight='bold')
    ax.set_ylabel('Filtered variance', fontsize=12, fontweight='bold')
    ax.set_title(f"Mexican Hat variance spectrum\n"
                 f"log M={row['logmbh']:.2f}, log λ={row['logedd']:.2f}, "
                 f"z={row['z']:.2f}, N={row['n_epochs']}",
                 fontsize=11, fontweight='bold')
    ax.grid(ls=':', alpha=0.5)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------

if __name__ == "__main__":
    for band in BANDS_TO_PROCESS:
        df = process_band(band, base_dir, csv_meta_path)
        if len(df) == 0:
            print(f"No valid objects for band {band}; skipping.")
            continue

        # Save full results (list columns as CSV strings)
        df_save = df.copy()
        for col in ('scales', 'variance', 'variance_err',
                    'variance_obs', 'variance_noise', 'n_valid'):
            df_save[col] = df_save[col].apply(lambda x: ','.join(map(str, x)))
        df_save.to_csv(os.path.join(OUT_DIR, f"mh_results_band_{band}.csv"),
                       index=False)
        print(f"Saved {OUT_DIR}/mh_results_band_{band}.csv")

        # Example individual plot
        plot_individual_example(df.iloc[0],
                                os.path.join(OUT_DIR, f"mh_example_band_{band}.png"))

        # Stack by mass
        st_m = stack_variance(df, mass_edges, edd_edges, group_by='mass')
        if st_m:
            plot_variance_spectra(
                st_m,
                f"Mexican Hat variance spectrum by mass (band {band})",
                os.path.join(OUT_DIR, f"mh_stacked_mass_band_{band}.png"))
            rows = []
            for lbl, d in st_m.items():
                for j, s in enumerate(d['scales']):
                    rows.append({'bin_label': lbl, 'bin_center': d['bin_center'],
                                 'n_objects': d['n_objects'],
                                 'scale': s,
                                 'median_variance': d['median_variance'][j]})
            pd.DataFrame(rows).to_csv(
                os.path.join(OUT_DIR, f"mh_stacked_mass_band_{band}.csv"),
                index=False)

        # Stack by Eddington
        st_e = stack_variance(df, mass_edges, edd_edges, group_by='edd')
        if st_e:
            plot_variance_spectra(
                st_e,
                f"Mexican Hat variance spectrum by Eddington ratio (band {band})",
                os.path.join(OUT_DIR, f"mh_stacked_edd_band_{band}.png"))
            rows = []
            for lbl, d in st_e.items():
                for j, s in enumerate(d['scales']):
                    rows.append({'bin_label': lbl, 'bin_center': d['bin_center'],
                                 'n_objects': d['n_objects'],
                                 'scale': s,
                                 'median_variance': d['median_variance'][j]})
            pd.DataFrame(rows).to_csv(
                os.path.join(OUT_DIR, f"mh_stacked_edd_band_{band}.csv"),
                index=False)

    print("\nAll done.")


 Processing band: G
Loading metadata...
Found 2261 folders to scan.


Band g: 100%|██████████| 2261/2261 [12:48<00:00,  2.94it/s]


Processed 2200 objects for band g.
Saved mexican_hat_results/mh_results_band_g.csv
  saved mexican_hat_results\mh_stacked_mass_band_g.png
  saved mexican_hat_results\mh_stacked_edd_band_g.png

 Processing band: R
Loading metadata...
Found 2261 folders to scan.


Band r: 100%|██████████| 2261/2261 [15:11<00:00,  2.48it/s]


Processed 2216 objects for band r.
Saved mexican_hat_results/mh_results_band_r.csv
  saved mexican_hat_results\mh_stacked_mass_band_r.png
  saved mexican_hat_results\mh_stacked_edd_band_r.png

All done.


In [1]:
# ============================================================================
# MEXICAN HAT -> APPROXIMATE PSD, WITH STACKING BY MASS AND EDDINGTON
# ============================================================================
#
# The Mexican Hat gives Var(sigma), the band-integrated power at timescale
# sigma. To compare with a PSD P(f) from Petrecca/CARMA, we convert:
#
#     f_sigma  = 1 / (2 * pi * sigma)
#     P_approx = Var(sigma) * 2 * pi * sigma        [flux^2 / day]
#
# This is a *point estimate* of the PSD at f_sigma; it is NOT the full PSD,
# but it is the best single-frequency proxy one can extract from the MH
# filtered variance. The slope of log(P_approx) vs log(f_sigma) recovers
# the true PSD slope alpha (since Var ~ P * Delta_f, and Delta_f ~ f).
#
# We do the stacking in the same mass and Eddington bins as the previous
# cell, but now the stacked quantity is P_approx(f_sigma) instead of
# Var(sigma).
#
# Output directory: plots_mexican_hat_psd/
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# CONFIGURATION (same as before)
# ----------------------------------------------------------------------------
base_dir      = r"ztffp_final_data_2025"
csv_meta_path = r"wu_shen_selected_columns.csv"
BANDS_TO_PROCESS = ['g', 'r']

SCALES_REST_DAYS = [20, 30, 50, 75, 100, 150, 200, 300, 450, 600]

MIN_POINTS_IN_KERNEL = 5
MIN_POINTS_PER_SIDE  = 2
MIN_SPAN_FRACTION    = 0.4

Z_MIN, Z_MAX = 0.4, 0.5
mass_edges = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
edd_edges  = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5]

N_NOISE_SIMULATIONS = 20

PSD_OUT_DIR = "plots_mexican_hat_psd"
os.makedirs(PSD_OUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# MEXICAN HAT HELPERS (identical to the previous cell)
# ----------------------------------------------------------------------------
def mexican_hat_kernel(u, sigma):
    x = u / sigma
    return (1.0 - x**2) * np.exp(-0.5 * x**2)

def mh_variance_single(t, f, sigma):
    n = len(t)
    if n < MIN_POINTS_IN_KERNEL:
        return np.nan, 0
    dt = t[:, None] - t[None, :]
    K  = mexican_hat_kernel(dt, sigma)
    support = np.abs(dt) < 3.0 * sigma
    K2 = (K ** 2) * support
    norm_sq = np.sum(K2, axis=1)

    valid = np.zeros(n, dtype=bool)
    for i in range(n):
        if norm_sq[i] <= 0:
            continue
        sup_i = support[i]
        if np.sum(sup_i) < MIN_POINTS_IN_KERNEL:
            continue
        dt_row = dt[i]
        n_left  = np.sum((dt_row > 0) & (dt_row < 2.0*sigma) & sup_i)
        n_right = np.sum((dt_row < 0) & (dt_row > -2.0*sigma) & sup_i)
        if n_left < MIN_POINTS_PER_SIDE or n_right < MIN_POINTS_PER_SIDE:
            continue
        t_in = t[sup_i]
        span = t_in.max() - t_in.min() if len(t_in) > 0 else 0.0
        if span < 4.0 * sigma * MIN_SPAN_FRACTION:
            continue
        valid[i] = True

    n_valid = int(np.sum(valid))
    if n_valid < max(5, MIN_POINTS_IN_KERNEL):
        return np.nan, n_valid
    idx_valid = np.where(valid)[0]
    F = np.empty(n_valid, dtype=float)
    for k, i in enumerate(idx_valid):
        numer = np.sum(f * K[i] * support[i])
        F[k] = numer / np.sqrt(norm_sq[i])
    return np.mean(F**2) - np.mean(F)**2, n_valid

def mh_variance_with_noise(t, f, f_err, sigma, n_sim=N_NOISE_SIMULATIONS):
    var_obs, n_valid = mh_variance_single(t, f, sigma)
    if np.isnan(var_obs) or f_err is None:
        return var_obs, n_valid
    if np.all(f_err == 0) or np.all(np.isnan(f_err)):
        return var_obs, n_valid
    rng = np.random.default_rng(42)
    noise_vars = []
    for _ in range(n_sim):
        noise = rng.normal(0.0, f_err)
        v_n, _ = mh_variance_single(t, noise, sigma)
        if not np.isnan(v_n):
            noise_vars.append(v_n)
    if not noise_vars:
        return var_obs, n_valid
    var_corr = var_obs - float(np.median(noise_vars))
    if var_corr < 0:
        return np.nan, n_valid
    return var_corr, n_valid

def compute_mh_psd_spectrum(t, f, f_err, scales):
    """
    Returns a dict with, per scale:
        - variance  (corrected)
        - f_psd     (approx frequency = 1/(2*pi*sigma))
        - P_approx  (approx PSD = variance * 2*pi*sigma)
    """
    idx = np.argsort(t)
    t, f = t[idx], f[idx]
    if f_err is not None:
        f_err = f_err[idx]

    out = {
        'scales':    [],
        'variance':  [],
        'f_psd':     [],
        'P_approx':  [],
        'n_valid':   [],
    }
    for sigma in scales:
        v_corr, nv = mh_variance_with_noise(t, f, f_err, sigma)
        f_psd = 1.0 / (2.0 * np.pi * sigma)
        P_approx = np.nan if np.isnan(v_corr) else v_corr * 2.0 * np.pi * sigma
        out['scales'].append(sigma)
        out['variance'].append(v_corr)
        out['f_psd'].append(f_psd)
        out['P_approx'].append(P_approx)
        out['n_valid'].append(nv)
    return out

# ----------------------------------------------------------------------------
# BAND PROCESSING (same filters, but now returns PSD columns)
# ----------------------------------------------------------------------------
def process_band_psd(band, base_dir, csv_meta_path):
    print(f"\n{'='*60}\n Processing band: {band.upper()}\n{'='*60}")
    print("Loading metadata...")
    meta = pd.read_csv(csv_meta_path.replace("\\", "/"))
    ra_cat    = meta['RA'].values
    dec_cat   = meta['DEC'].values
    logmbh    = meta['LOGMBH'].values
    logedd    = meta['LOGLEDD_RATIO'].values
    z_col     = 'Z_DR16Q' if 'Z_DR16Q' in meta.columns else 'Z'
    z_cat     = meta[z_col].values

    folders = [f for f in os.listdir(base_dir)
               if os.path.isdir(os.path.join(base_dir, f))
               and "plots" not in f and "output" not in f]

    results = []
    for folder in tqdm(folders, desc=f"Band {band}"):
        try:
            ra_f, dec_f = map(float, folder.split('_'))
        except ValueError:
            continue
        dist = np.sqrt((ra_cat - ra_f)**2 + (dec_cat - dec_f)**2)
        idx = int(np.argmin(dist))
        if dist[idx] > 1e-4:
            continue
        m, e, z = logmbh[idx], logedd[idx], z_cat[idx]
        if np.isnan(m) or np.isnan(e) or np.isnan(z):
            continue
        if not (Z_MIN <= z <= Z_MAX):
            continue
        if not (mass_edges[0] <= m < mass_edges[-1]):
            continue
        if not (edd_edges[0] <= e < edd_edges[-1]):
            continue

        txt_files = glob.glob(os.path.join(base_dir, folder,
                                           f"*_ztf_{band}_filtered.txt"))
        if not txt_files:
            continue
        try:
            df = pd.read_csv(txt_files[0])
        except Exception:
            continue
        if len(df) < 10 or 'mjd' not in df.columns or 'flux' not in df.columns:
            continue

        df = df.sort_values('mjd')
        t_obs = df['mjd'].values
        f_raw = df['flux'].values
        f_err_raw = None
        for col in ('flux_err', 'flux_error', 'err'):
            if col in df.columns:
                f_err_raw = df[col].values
                break

        med = np.median(f_raw)
        if med <= 0 or np.isnan(med):
            continue
        f_norm = (f_raw - med) / med
        f_err_norm = (f_err_raw / med) if f_err_raw is not None else None

        t_rest = t_obs / (1.0 + z)
        t_rest = t_rest - t_rest.min()

        spec = compute_mh_psd_spectrum(t_rest, f_norm, f_err_norm,
                                       SCALES_REST_DAYS)
        results.append({
            'ra': ra_f, 'dec': dec_f,
            'logmbh': m, 'logedd': e, 'z': z,
            'n_epochs': len(df),
            't_baseline_rest': float(t_rest.max()),
            **spec,
        })

    print(f"Processed {len(results)} objects for band {band}.")
    return pd.DataFrame(results)

# ----------------------------------------------------------------------------
# STACKING: median P_approx per (bin, f_psd)
# ----------------------------------------------------------------------------
def stack_psd(df, edges, col, group_by, min_objects=3):
    """
    Returns a dict: {bin_label: {f_psd, median_P, n_objects, bin_center}}.
    """
    out = {}
    fmt = "logM {:.1f}-{:.1f}" if group_by == 'mass' else "logλ {:.1f}-{:.1f}"

    scales = list(df['scales'].iloc[0])
    f_psd_arr = list(df['f_psd'].iloc[0])

    for k in range(len(edges) - 1):
        lo, hi = edges[k], edges[k + 1]
        sub = df[(df[col] >= lo) & (df[col] < hi)]
        if len(sub) < min_objects:
            continue

        med_P = []
        for j in range(len(scales)):
            vals = []
            for _, row in sub.iterrows():
                p = row['P_approx'][j]
                if not np.isnan(p) and p > 0:
                    vals.append(p)
            med_P.append(np.median(vals) if len(vals) >= min_objects else np.nan)

        out[fmt.format(lo, hi)] = {
            'f_psd': f_psd_arr,
            'median_P': med_P,
            'n_objects': len(sub),
            'bin_center': 0.5 * (lo + hi),
        }
    return out

# ----------------------------------------------------------------------------
# PLOTTING: log(P) vs log(f) for each stacked bin
# ----------------------------------------------------------------------------
def plot_stacked_psd(stacked, title, out_path):
    fig, ax = plt.subplots(figsize=(9, 6))
    cmap = plt.get_cmap('viridis')
    n = len(stacked)
    for i, (label, d) in enumerate(stacked.items()):
        color = cmap(i / max(1, n - 1))
        f = np.array(d['f_psd'], dtype=float)
        p = np.array(d['median_P'], dtype=float)
        ok = (~np.isnan(p)) & (p > 0)
        if np.sum(ok) < 2:
            continue
        ax.plot(f[ok], p[ok], marker='o', lw=2, ms=6, color=color,
                label=f"{label} (N={d['n_objects']})")
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Approx. rest-frame frequency $f = 1/(2\pi\sigma)$ [day$^{-1}$]',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel(r'Approx. PSD $P(f) \approx \mathrm{Var}(\sigma)\cdot 2\pi\sigma$',
                  fontsize=13, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(ls=':', alpha=0.5, which='both')
    ax.legend(fontsize=9, loc='best', framealpha=0.9)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"  saved {out_path}")

# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    for band in BANDS_TO_PROCESS:
        df = process_band_psd(band, base_dir, csv_meta_path)
        if len(df) == 0:
            print(f"No valid objects for band {band}; skipping.")
            continue

        # Save per-object results
        df_save = df.copy()
        for col in ('scales', 'variance', 'f_psd', 'P_approx', 'n_valid'):
            df_save[col] = df_save[col].apply(lambda x: ','.join(map(str, x)))
        df_save.to_csv(os.path.join(PSD_OUT_DIR, f"mh_psd_objects_band_{band}.csv"),
                       index=False)

        # Stack by mass
        st_m = stack_psd(df, mass_edges, 'logmbh', 'mass')
        if st_m:
            plot_stacked_psd(
                st_m,
                f"Mexican Hat approx. PSD by mass (band {band})",
                os.path.join(PSD_OUT_DIR, f"mh_psd_stacked_mass_band_{band}.png"))
            rows = []
            for lbl, d in st_m.items():
                for j, f in enumerate(d['f_psd']):
                    rows.append({'bin_label': lbl, 'bin_center': d['bin_center'],
                                 'n_objects': d['n_objects'],
                                 'f_psd': f,
                                 'median_P': d['median_P'][j]})
            pd.DataFrame(rows).to_csv(
                os.path.join(PSD_OUT_DIR, f"mh_psd_stacked_mass_band_{band}.csv"),
                index=False)

        # Stack by Eddington
        st_e = stack_psd(df, edd_edges, 'logedd', 'edd')
        if st_e:
            plot_stacked_psd(
                st_e,
                f"Mexican Hat approx. PSD by Eddington ratio (band {band})",
                os.path.join(PSD_OUT_DIR, f"mh_psd_stacked_edd_band_{band}.png"))
            rows = []
            for lbl, d in st_e.items():
                for j, f in enumerate(d['f_psd']):
                    rows.append({'bin_label': lbl, 'bin_center': d['bin_center'],
                                 'n_objects': d['n_objects'],
                                 'f_psd': f,
                                 'median_P': d['median_P'][j]})
            pd.DataFrame(rows).to_csv(
                os.path.join(PSD_OUT_DIR, f"mh_psd_stacked_edd_band_{band}.csv"),
                index=False)

    print("\nAll done. PSD stacked plots saved in:", PSD_OUT_DIR)


 Processing band: G
Loading metadata...


FileNotFoundError: [Errno 2] No such file or directory: 'wu_shen_selected_columns.csv'